In [96]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [97]:
df = pd.read_csv('housing.csv')
df.tail()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND
20639,-121.24,39.37,16.0,2785.0,616.0,1387.0,530.0,2.3886,89400.0,INLAND


In [98]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [100]:
df.isna().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [101]:
df.shape

(20640, 10)

In [102]:
df['ocean_proximity'].unique()

<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

In [103]:
#Dropping data since it is less than 1%
cleaned_df = df.dropna()

In [104]:
cleaned_df.isna().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

In [105]:
cleaned_df.shape

(20433, 10)

In [106]:
df['ocean_proximity'].unique()

<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

In [107]:
#Created 5 new columns to represent ocean_proximity since we cant perform ml in textual format
cleaned_df['is_near_bay'] = np.where(cleaned_df['ocean_proximity'].str.contains('NEAR BAY'), 1, 0)
cleaned_df['is_<1h_ocean'] = np.where(cleaned_df['ocean_proximity'].str.contains('<1H OCEAN'), 1, 0)
cleaned_df['is_inland'] = np.where(cleaned_df['ocean_proximity'].str.contains('INLAND'), 1, 0)
cleaned_df['is_near_ocean'] = np.where(cleaned_df['ocean_proximity'].str.contains('NEAR OCEAN'), 1, 0)
cleaned_df['is_island'] = np.where(cleaned_df['ocean_proximity'].str.contains('ISLAND'), 1, 0)

In [108]:
#Dropping the initial ocean proximity since we have created 5 new columnns for each value
cleaned_df = cleaned_df.drop(columns=['ocean_proximity'])

In [109]:
#Droppped median_house_value from x to seperate it into x train and test
X = cleaned_df.drop(columns=['median_house_value'])
y = cleaned_df[['median_house_value']]

In [136]:
#Seperated the x to train and test in 80% and 20% with random state 42 to shuffle the data and 42 for reproducibility
X_train = X.sample(frac=0.8, random_state=42)
X_test = X.drop(X_train.index)

y_train = y.sample(frac=0.8, random_state=42)
y_test = y.drop(X_train.index)

In [ ]:
#Performed Standardaization scaling
columns_to_scale = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
scalled_X_train = (X_train[columns_to_scale] - X_train[columns_to_scale].mean()) / X_train[columns_to_scale].std()
scalled_X_test = (X_test[columns_to_scale] - X_test[columns_to_scale].mean()) / X_test[columns_to_scale].std()

In [ ]:
#Replaced the value of scaled to the original train and test dataset without scaling certain columns -> ocean proximity
X_train[columns_to_scale] = scalled_X_train[columns_to_scale]
X_test[columns_to_scale] = scalled_X_test[columns_to_scale]